# Pipeline GNN (EdgeSAGE) — bản chạy trên Kaggle

**Chỉ cần upload đúng file notebook này.** Các cell `%%writefile` sẽ tự ghi code ra
`/kaggle/working/` khi chạy, nên không phải tạo Dataset riêng cho file `.py`.

## Chuẩn bị

1. **Add Data** -> `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`
   (chứa `HI-Small_Trans.csv`, ~475MB). Đây là dataset DUY NHẤT cần thêm.
2. **Settings** -> Accelerator = **GPU T4 x2**, Internet = **On**.
3. Run All.

## Vì sao dùng `%%writefile` + `!python` thay vì viết thẳng logic vào cell

Mỗi bước chạy trong một tiến trình riêng, thoát là hệ điều hành thu hồi toàn bộ RAM.
Bước `feature_node` có đỉnh RAM khoảng 12GB; nếu chạy chung một kernel với các bước
sau thì bộ nhớ cộng dồn và dễ chết kernel. Code vẫn nằm nguyên trong cell để đọc và sửa.

## Luồng dữ liệu

```
HI-Small_Trans.csv  (/kaggle/input, read-only)
  1 temporal_split_index  -> HI-Small_Trans_split_index.csv
  2 feature_transaction   -> transaction_features.parquet
  3 feature_node          -> node_edge_features.parquet
  4 feature_edge          -> edge_attr_{train,val,test}.parquet
  5 assemble_txn          -> txn_matrix_{train,val,test}.parquet
  6 build_graph           -> graphs.pt, txn_nodes.npy
  7 train_gnn edge/noedge -> results.csv, scores/*.npy
```

File sinh ra nằm ở `/kaggle/working/dataset_high/`, tổng khoảng 3.8GB.
Ước lượng thời gian toàn bộ: **1-1.5 giờ** trên T4.

## Vì sao mọi lệnh chạy đều có `PYTHONPATH=/kaggle/working`

Kaggle không đưa thư mục chứa script vào `sys.path` như Python mặc định, nên
`from paths import ...` báo `ModuleNotFoundError: No module named 'paths'`.
Chỉ định `PYTHONPATH` tường minh là cách chắc chắn nhất, và mỗi cell vẫn chạy
độc lập được, không phụ thuộc cell nào trước đó.


## 0. Cài đặt

In [ ]:
# torch_geometric không có sẵn trên Kaggle
!pip install -q torch_geometric

## 1. Ghi code ra /kaggle/working

Sửa code thì sửa ngay trong cell rồi chạy lại đúng cell đó.

In [ ]:
%%writefile /kaggle/working/paths.py
"""
Đường dẫn dùng chung cho toàn pipeline.

Tự nhận môi trường:
  - Kaggle : đọc CSV gốc từ /kaggle/input (read-only), ghi kết quả vào /kaggle/working
  - Local  : đọc và ghi trong thư mục dataset_high, y hệt bản gốc

Ghi đè bằng biến môi trường khi cần:
  AML_RAW_DIR  - thư mục chứa HI-Small_Trans.csv
  AML_OUT_DIR  - thư mục ghi các file trung gian
"""
import glob
import os

_ON_KAGGLE = os.path.isdir("/kaggle/working")


def _detect_raw_dir() -> str:
    env = os.environ.get("AML_RAW_DIR")
    if env:
        return env
    if _ON_KAGGLE:
        hits = sorted(glob.glob("/kaggle/input/**/HI-Small_Trans.csv", recursive=True))
        if not hits:
            raise FileNotFoundError(
                "Không thấy HI-Small_Trans.csv trong /kaggle/input.\n"
                "Vào Add Data -> thêm dataset "
                "'ealtman2019/ibm-transactions-for-anti-money-laundering-aml',\n"
                "hoặc đặt biến môi trường AML_RAW_DIR trỏ tới thư mục chứa file."
            )
        return os.path.dirname(hits[0])
    return "dataset_high"


RAW_DIR = _detect_raw_dir()
OUT_DIR = os.environ.get("AML_OUT_DIR") or (
    "/kaggle/working/dataset_high" if _ON_KAGGLE else "dataset_high"
)
WORK_DIR = os.path.dirname(os.path.abspath(OUT_DIR))

# --- input gốc (read-only trên Kaggle) ---
TRANS_CSV = os.path.join(RAW_DIR, "HI-Small_Trans.csv")

# --- file trung gian (ghi được) ---
SPLIT_CSV = os.path.join(OUT_DIR, "HI-Small_Trans_split_index.csv")
TX_FEAT = os.path.join(OUT_DIR, "transaction_features.parquet")
NODE_FEAT = os.path.join(OUT_DIR, "node_edge_features.parquet")
EDGE_ATTR = os.path.join(OUT_DIR, "edge_attr_{}.parquet")
TXN_MATRIX = os.path.join(OUT_DIR, "txn_matrix_{}.parquet")
GRAPHS_PT = os.path.join(OUT_DIR, "graphs.pt")
TXN_NODES = os.path.join(OUT_DIR, "txn_nodes.npy")

# --- output kết quả ---
RESULTS_CSV = os.path.join(WORK_DIR, "results.csv")
SCORES_DIR = os.path.join(WORK_DIR, "scores")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SCORES_DIR, exist_ok=True)


def describe() -> str:
    return (f"môi trường : {'Kaggle' if _ON_KAGGLE else 'local'}\n"
            f"RAW_DIR    : {RAW_DIR}\n"
            f"OUT_DIR    : {OUT_DIR}\n"
            f"WORK_DIR   : {WORK_DIR}")


if __name__ == "__main__":
    print(describe())
    print(f"\nTRANS_CSV tồn tại: {os.path.exists(TRANS_CSV)}")

In [ ]:
%%writefile /kaggle/working/temporal_split_index.py
import pandas as pd

from paths import TRANS_CSV, SPLIT_CSV


def main():
    # ép kiểu phòng trường hợp pandas đọc thành int và có số 0 đầu sẽ bỏ số 0 đầu
    # sẽ làm sai dữ liệu khi tạo node
    dtype = {"From Bank": str, "To Bank": str, "Account": str, "Account.1": str}
    df_trans = pd.read_csv(TRANS_CSV, dtype=dtype)

    # sắp xếp dataframe theo timestamp
    order = df_trans["Timestamp"].sort_values(kind="mergesort").index
    df_trans = df_trans.iloc[order].reset_index(drop=True)

    # chia dữ liệu theo train/test/val
    n = len(df_trans)
    t1 = int(n * 0.6)
    t2 = int(n * 0.8)
    ts1 = df_trans["Timestamp"].iloc[t1]
    ts2 = df_trans["Timestamp"].iloc[t2]

    # gán nhãn train/test/val cho tập dữ liệu mới và xuất file
    df_trans["split"] = "test"
    df_trans.loc[df_trans["Timestamp"] < ts2, "split"] = "val"
    df_trans.loc[df_trans["Timestamp"] < ts1, "split"] = "train"
    assert df_trans[df_trans.split == "train"]["Timestamp"].max() < df_trans[df_trans.split == "val"]["Timestamp"].min(), \
        "Leakage: Timestamp chong lan giua train va val"
    assert df_trans[df_trans.split == "val"]["Timestamp"].max() < df_trans[df_trans.split == "test"]["Timestamp"].min(), \
        "Leakage: Timestamp chong lan giua val va test"

    # xuất file csv
    df_trans.to_csv(SPLIT_CSV, index=False)
    print(f"Đã lưu: {SPLIT_CSV}")
    print(df_trans["split"].value_counts())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/feature_transaction.py
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from paths import SPLIT_CSV, TX_FEAT

SRC = SPLIT_CSV
OUT = TX_FEAT

SCALE_COLS = ["amt_paid_log"]


def load_data():
    dt = {"From Bank": str, "Account": str, "To Bank": str, "Account.1": str, "split": str}
    df = pd.read_csv(SRC, dtype=dt, engine="pyarrow")
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M")
    df["src"] = df["From Bank"] + " | " + df["Account"]
    df["dest"] = df["To Bank"] + " | " + df["Account.1"]
    return df


def build_transaction_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["amt_paid_log"] = np.log1p(df["Amount Paid"])
    out["is_cross_bank"] = (df["From Bank"] != df["To Bank"]).astype("int8")
    out["is_cross_currency"] = (df["Receiving Currency"] != df["Payment Currency"]).astype("int8")
    out["is_round_1000"] = (df["Amount Paid"] % 1000 == 0).astype("int8")
    out["is_round_100"] = (df["Amount Paid"] % 100 == 0).astype("int8")
    out["is_self_loop"] = (df["src"] == df["dest"]).astype("int8")

    # --- Thời gian ---
    hour = df["Timestamp"].dt.hour.astype("int16")
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    day_of_week = df["Timestamp"].dt.dayofweek.astype("int16")
    out["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    out["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    # --- One-hot (Tối ưu RAM bằng cách ép dtype ngay từ đầu) ---
    pf = pd.get_dummies(df["Payment Format"], prefix="pf", dtype="int8")
    ccy = pd.get_dummies(df["Payment Currency"], prefix="ccy", dtype="int8")
    out = pd.concat([out, pf, ccy], axis=1)

    # --- Khóa join và nhãn (Để kiểu string/object để ghi Parquet không lỗi) ---
    out["src"] = df["src"]
    out["dest"] = df["dest"]
    out["Is Laundering"] = df["Is Laundering"].astype("int8")
    out["split"] = df["split"]
    return out


def scale_features(feat: pd.DataFrame, scale_cols=SCALE_COLS):
    tr = feat["split"] == "train"
    scaler = StandardScaler().fit(feat.loc[tr, scale_cols].values)

    out = feat.copy()
    out[scale_cols] = scaler.transform(feat[scale_cols].values)
    return out, scaler


def main():
    print("--- 1. Loading data ---")
    df = load_data()

    print("--- 2. Building features ---")
    feat = build_transaction_features(df)

    print("--- 3. Scaling features (Train-only fit) ---")
    feat, scaler = scale_features(feat)

    os.makedirs(os.path.dirname(OUT), exist_ok=True)

    print("--- 4. Saving features ---")
    try:
        path = OUT
        feat.to_parquet(path, index=False)
        print(f"[Success] Đã lưu dạng Parquet: {path}")
    except Exception as e:
        path = OUT.replace(".parquet", ".csv")
        feat.to_csv(path, index=False)
        print(f"[Warn] Không lưu được Parquet ({e}). Đã fallback lưu CSV: {path}")

    # In thông tin kiểm tra data split
    n_feat = feat.shape[1] - 4  # trừ src, dest, Is Laundering, split
    print(f"\nThống kê: rows={len(feat):,} | n_features={n_feat}")

    g = feat.groupby("split", observed=True)["Is Laundering"]
    rep = pd.DataFrame({"n": g.size(), "n_pos": g.sum(), "pos_%": (g.mean() * 100).round(3)})
    print(rep)

    # đọc lại để xác nhận file ghi đúng (chỉ 2 cột, tránh nạp lại toàn bộ 154MB)
    f = pd.read_parquet(OUT, columns=["split", "amt_paid_log"])
    print("mean(amt_paid_log) trên train:", f[f.split == "train"]["amt_paid_log"].mean())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/feature_node.py
import gc
import os

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from paths import SPLIT_CSV, NODE_FEAT

src = SPLIT_CSV
out = NODE_FEAT


def load_data():
    dt = {"From Bank": str, "Account": str, "To Bank": str, "Account.1": str, "split": str}
    df = pd.read_csv(src, dtype=dt, engine="pyarrow")
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M")
    df["src"] = df["From Bank"] + " | " + df["Account"]
    df["dest"] = df["To Bank"] + " | " + df["Account.1"]
    df["ori_idx"] = np.arange(len(df))
    df = df.sort_values(["Timestamp", "ori_idx"], kind="mergesort").reset_index(drop=True)
    df["time"] = df["Timestamp"].astype("int64")
    df["is_cross_bank"] = df["From Bank"] != df["To Bank"]
    df["is_cross_curcy"] = df["Payment Currency"] != df["Receiving Currency"]
    df["is_round"] = df["Amount Paid"] % 1000 == 0
    return df


def asof_block(df, group_cols, amt_col, partner_col, bank_col, curcy_col, pf_encoding, prefix, log_col_out):
    # mean/sum/count
    out = pd.DataFrame(index=df.index)
    keys = [df[c] for c in group_cols]
    g = df.groupby(group_cols, sort=False)
    x = df[amt_col]
    cnt = g.cumcount()
    out[f"{prefix}cnt"] = cnt
    sum_prior = g[amt_col].cumsum() - x
    out[f"{prefix}sum"] = sum_prior
    mean_prior = (sum_prior / cnt).fillna(0)
    out[f"{prefix}mean"] = mean_prior
    # phương sai
    x2 = x ** 2
    sumsq_prior = x2.groupby(keys, sort=False).cumsum() - x2
    meansq_prior = (sumsq_prior / cnt).fillna(0)
    var_prior = meansq_prior - mean_prior ** 2
    std_prior = np.sqrt(np.maximum(var_prior, 0.0))
    out[f"{prefix}std"] = std_prior
    # min/max
    out[f"{prefix}max"] = g[amt_col].cummax().groupby(keys, sort=False).shift(1).fillna(0)
    out[f"{prefix}min"] = g[amt_col].cummin().groupby(keys, sort=False).shift(1).fillna(0)

    def nunique_prior(sub_col):
        flag = (~df.duplicated(subset=group_cols + [sub_col])).astype("float32")
        count = flag.groupby(keys, sort=False).cumsum()
        return count - flag

    if partner_col is not None:
        out[f"{prefix}cnt_partner"] = nunique_prior(partner_col)
    if bank_col is not None:
        out[f"{prefix}cnt_bank"] = nunique_prior(bank_col)
    if curcy_col is not None:
        out[f"{prefix}cnt_curcy"] = nunique_prior(curcy_col)
    for fname, fcol in [("cross_bank", "is_cross_bank"), ("cross_curcy", "is_cross_curcy"),
                        ("round", "is_round")]:
        prior_f = df[fcol].groupby(keys, sort=False).cumsum() - df[fcol]
        out[f"{prefix}ratio_{fname}"] = (prior_f / cnt).fillna(0).astype("float32")
    for col in pf_encoding.columns:
        prior_d = pf_encoding[col].groupby(keys, sort=False).cumsum() - pf_encoding[col]
        out[f"{prefix}ratio_{col}"] = (prior_d / cnt).fillna(0).astype("float32")
    first_seen = df["time"].groupby(keys, sort=False).cummin()
    active_time = ((df["time"] - first_seen) / 1e9).fillna(0)
    out[f"{prefix}active_time"] = active_time
    out[f"{prefix}tx_per_day"] = (cnt / (active_time / 86400).clip(lower=1.0)).fillna(0)
    out[f"{prefix}first_seen"] = (cnt == 0).astype("int8")
    log_col_out += [f"{prefix}cnt", f"{prefix}sum", f"{prefix}mean", f"{prefix}std",
                    f"{prefix}max", f"{prefix}min", f"{prefix}tx_per_day", f"{prefix}active_time"]
    for c in (partner_col, curcy_col, bank_col):
        if c is not None:
            suffix = {partner_col: "partner", bank_col: "bank", curcy_col: "curcy"}[c]
            log_col_out.append(f"{prefix}cnt_{suffix}")
    return out


def build_entity_features(df):
    pf_encoding = pd.get_dummies(df["Payment Format"], dtype="int8")
    log_cols = []
    src_block = asof_block(df, ["src"], "Amount Paid", partner_col="dest", bank_col="To Bank",
                           curcy_col="Payment Currency", pf_encoding=pf_encoding,
                           prefix="src_", log_col_out=log_cols)
    dest_block = asof_block(df, ["dest"], "Amount Received", partner_col="src", bank_col="From Bank",
                            curcy_col="Receiving Currency", pf_encoding=pf_encoding,
                            prefix="dest_", log_col_out=log_cols)
    pair_block = asof_block(df, ["src", "dest"], "Amount Paid", partner_col=None, bank_col=None,
                            curcy_col=None, pf_encoding=pf_encoding,
                            prefix="pair_", log_col_out=log_cols)
    asof = pd.concat([src_block, dest_block, pair_block], axis=1)
    # giải phóng sớm: 3 block này chiếm ~2.5GB, không cần giữ sau khi concat
    del src_block, dest_block, pair_block, pf_encoding
    gc.collect()
    asof["split"] = df["split"].values
    asof["ori_idx"] = df["ori_idx"].values
    return asof, log_cols


def verify_asof(df, asof):
    verify = True
    first_row = ~df.duplicated(subset=["src"], keep="first")
    check_first = (asof.loc[first_row, "src_cnt"] == 0).all() and (asof.loc[first_row, "src_first_seen"] == 1).all() \
        and (asof.loc[first_row, "src_sum"] == 0).all()
    print(f"kiểm tra rò rỉ dữ liệu: {check_first} {'PASS' if check_first else 'lỗi check first'}")
    verify &= check_first
    last_row = ~df.duplicated(subset=["src"], keep="last")
    whole_window_count = df.groupby("src")["src"].transform("size")
    check_last = (asof.loc[last_row, "src_cnt"] == whole_window_count.loc[last_row] - 1).all()
    print(f"kiểm tra toàn vẹn: {'PASS' if check_last else 'lỗi check last'}")
    verify &= check_last
    test_rows = df.index[df["split"] == "test"]
    idx = test_rows[len(test_rows) // 2]
    row = df.loc[idx]
    prior_mask = (df["src"] == row["src"]) & ((df["time"] < row["time"]) |
                                              ((df["time"] == row["time"]) & (df["ori_idx"] < row["ori_idx"])))
    manual_cnt = int(prior_mask.sum())
    manual_sum = df.loc[prior_mask, "Amount Paid"].sum()
    got_cnt = asof.loc[idx, "src_cnt"]
    got_sum = asof.loc[idx, "src_sum"]
    spot_check = (manual_cnt == got_cnt) and np.isclose(manual_sum, got_sum)
    print(f"kiểm tra ngẫu nhiên: index={idx}: count ={manual_cnt} vs count.1={got_cnt},"
          f"sum={manual_sum:.2f} vs sum.1={got_sum:.2f}")
    verify &= spot_check
    del first_row, last_row, whole_window_count, prior_mask
    gc.collect()
    if not verify:
        raise AssertionError("check fail sau khi kiểm tra logic")
    print("PASS ALL")


def scale_feature(asof, log_cols):
    out = asof.copy()
    out[log_cols] = np.log1p(out[log_cols].clip(lower=0))
    is_train = out["split"] == "train"
    scaler = StandardScaler().fit(out.loc[is_train, log_cols].values)
    out[log_cols] = scaler.transform(out[log_cols].values)
    return out, scaler


def main():
    df = load_data()
    asof, log_cols = build_entity_features(df)
    verify_asof(df, asof)
    asof, scaler = scale_feature(asof, log_cols)
    gc.collect()
    asof = asof.sort_values("ori_idx").drop(columns="ori_idx")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    asof.to_parquet(out, index=False)
    print(f"Success Đã lưu: {out} shape={asof.shape}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/feature_edge.py
import gc

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from feature_node import load_data      # đã sort Timestamp + tạo src/dest + cờ
from paths import EDGE_ATTR

EDGE_LOG1P_COLS = [
    "num_tx", "total_paid", "mean_paid", "std_paid",
    "min_paid", "max_paid", "active_day", "tx_per_day",
]

# name -> (cửa sổ tính FEATURE = lagged, cửa sổ định nghĩa TẬP CẠNH = lũy tiến Altman)
WINDOWS = {
    "train": (["train"],        ["train"]),
    "val":   (["train"],        ["train", "val"]),
    "test":  (["train", "val"], ["train", "val", "test"]),
}


def aggregate_edges(win: pd.DataFrame, formats, currencies):
    """Thống kê cạnh gộp trên cửa sổ NGUỒN. Không đụng tới 'Is Laundering'."""
    g = win.groupby(["src", "dest"], sort=False)
    agg = g.agg(
        num_tx          = ("Amount Paid", "size"),
        total_paid      = ("Amount Paid", "sum"),
        mean_paid       = ("Amount Paid", "mean"),
        std_paid        = ("Amount Paid", "std"),
        min_paid        = ("Amount Paid", "min"),
        max_paid        = ("Amount Paid", "max"),
        cross_ccy_ratio = ("is_cross_curcy", "mean"),
        round_ratio     = ("is_round", "mean"),
        time_min        = ("Timestamp", "min"),
        time_max        = ("Timestamp", "max"),
    )
    agg["std_paid"] = agg["std_paid"].fillna(0.0)          # cạnh 1-tx: ddof=1 -> NaN
    agg["active_day"] = (agg["time_max"] - agg["time_min"]).dt.days.clip(lower=1)
    agg["tx_per_day"] = agg["num_tx"] / agg["active_day"]
    agg = agg.drop(columns=["time_min", "time_max"])

    keys = [win["src"], win["dest"]]
    for col, vocab, prefix in [("Payment Format", formats,    "pf"),
                               ("Payment Currency", currencies, "ccy")]:
        d = pd.get_dummies(win[col]).astype("float32").reindex(columns=vocab, fill_value=0.0)
        prop = d.groupby(keys, sort=False).mean()
        prop.columns = [f"{prefix}_{c.replace(' ', '_')}" for c in vocab]
        agg = agg.join(prop)
        del d, prop
        gc.collect()
    return agg


def build_window(df, src_splits, graph_splits, formats, currencies):
    src_win   = df[df["split"].isin(src_splits)]
    graph_win = df[df["split"].isin(graph_splits)]

    # TẬP CẠNH lấy từ cửa sổ lũy tiến. is_cross_bank / is_self_loop là hàm thuần của
    # (src, dest) -> lấy ở đây không leak vì không đụng nội dung giao dịch.
    edges = graph_win.groupby(["src", "dest"], sort=False).agg(
        is_cross_bank = ("is_cross_bank", "max"),
        is_self_loop  = ("is_self_loop",  "max"),
    ).astype("int8")

    agg = aggregate_edges(src_win, formats, currencies)

    feat = edges.join(agg, how="left")                     # cạnh chưa có trong nguồn -> NaN
    feat["seen_before"] = feat["num_tx"].notna().astype("int8")
    del src_win, graph_win, edges, agg
    gc.collect()
    return feat.fillna(0.0)


def scale_edges(feats, log_cols):
    """log1p + StandardScaler fit CHỈ trên bảng 'train' (tính từ cửa sổ train)."""
    for name in feats:
        feats[name][log_cols] = np.log1p(feats[name][log_cols].clip(lower=0))
    scaler = StandardScaler().fit(feats["train"][log_cols].values)
    for name in feats:
        feats[name][log_cols] = scaler.transform(feats[name][log_cols].values)
    return feats, scaler


def verify(df, feats, formats, currencies):
    # 1. không có cột nào suy từ nhãn
    banned = {"is_edge_mule", "is_mule", "Is Laundering"}
    for name, f in feats.items():
        assert not (banned & set(f.columns)), f"{name} còn cột suy từ nhãn"

    # 2. edge_attr val chỉ phụ thuộc train: xáo Amount Paid trong val -> bảng không đổi
    #    chỉ copy các cột build_window thực sự dùng, tránh nhân đôi cả df 5M dòng
    need = ["split", "src", "dest", "Timestamp", "Amount Paid", "Payment Format",
            "Payment Currency", "is_cross_bank", "is_cross_curcy", "is_round", "is_self_loop"]
    shuffled = df[need].copy()
    m = shuffled["split"] == "val"
    rng = np.random.default_rng(0)
    shuffled.loc[m, "Amount Paid"] = rng.permutation(shuffled.loc[m, "Amount Paid"].values)
    ref = build_window(shuffled, *WINDOWS["val"], formats, currencies)
    base = build_window(df, *WINDOWS["val"], formats, currencies)
    assert ref.equals(base), "edge_attr val ĐANG phụ thuộc dữ liệu val -> leak"
    print("  PASS: edge_attr val bất biến khi xáo dữ liệu val")
    del shuffled, ref, base
    gc.collect()

    # 3. cold-start: tỉ lệ cạnh chưa từng thấy
    for name, f in feats.items():
        r = 1 - f["seen_before"].mean()
        print(f"  {name:5s}: {len(f):>9,} cạnh | cold-start {r*100:5.2f}%")


def main():
    df = load_data()
    df["is_self_loop"] = (df["src"] == df["dest"]).astype("int8")
    # vocab chốt trên train để tập cột ổn định giữa 3 file
    tr = df[df["split"] == "train"]
    formats    = sorted(tr["Payment Format"].unique())
    currencies = sorted(tr["Payment Currency"].unique())
    del tr
    gc.collect()

    feats = {n: build_window(df, *w, formats, currencies) for n, w in WINDOWS.items()}
    cols = list(feats["train"].columns)
    feats = {n: f[cols] for n, f in feats.items()}          # khoá thứ tự cột

    verify(df, feats, formats, currencies)
    feats, _ = scale_edges(feats, EDGE_LOG1P_COLS)

    for name, f in feats.items():
        p = EDGE_ATTR.format(name)
        f.reset_index().to_parquet(p, index=False)
        print(f"  đã lưu {p} shape={f.shape}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/assemble_txn.py
import pandas as pd, numpy as np, gc
import pyarrow.parquet as pq

from paths import TX_FEAT, NODE_FEAT, TXN_MATRIX

tx_path   = TX_FEAT
asof_path = NODE_FEAT
label     = "Is Laundering"
key_cols  = ["src", "dest"]
leak      = ["is_mule", "is_edge_mule"]


def feat_cols(path, drop):
    return [c for c in pq.ParquetFile(path).schema_arrow.names if c not in drop]


def main():
    tx_cols   = feat_cols(tx_path,   set(key_cols + [label, "split"]))
    asof_cols = feat_cols(asof_path, {"split", "ori_idx"})

    meta_tx = pd.read_parquet(tx_path, columns=[label, "split"])

    y     = meta_tx[label].to_numpy(dtype="int8")
    split = meta_tx["split"].to_numpy()
    n     = len(y)
    del meta_tx; gc.collect()

    names = tx_cols + asof_cols
    X = np.empty((n, len(names)), dtype="float32")
    j = 0
    for path, cols in ((tx_path, tx_cols), (asof_path, asof_cols)):
        pf = pq.ParquetFile(path)
        for c in cols:
            v = pf.read(columns=[c]).column(0).to_numpy(zero_copy_only=False)
            v = v.astype("float32", copy=False)
            X[:, j] = v; j += 1
            del v
        del pf; gc.collect()

    for s in ["train", "val", "test"]:
        m   = split == s
        matrix = pd.DataFrame(X[m], columns=names)
        matrix[label] = y[m]
        out = TXN_MATRIX.format(s)
        matrix.to_parquet(out, index=False)
        n_pos = int(matrix[label].sum())
        print(f"{s:5s}: {len(matrix):>9,} dòng | {len(names)} feature | "
              f"{n_pos:,} pos ({n_pos/len(matrix)*100:.3f}%) -> {out}")
        del matrix; gc.collect()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/build_graph.py
import os
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

from feature_node import load_data
from paths import EDGE_ATTR, GRAPHS_PT, TXN_NODES

GRAPH_SPLITS = {                       # cấu trúc lũy tiến chuẩn Altman
    "train": ["train"],
    "val":   ["train", "val"],
    "test":  ["train", "val", "test"],
}


def build_vocab(df):
    """Vocab node DÙNG CHUNG cho cả 3 graph -> embedding khớp index giữa các split."""
    a = df[["src", "From Bank"]].rename(columns={"src": "node", "From Bank": "bank"})
    b = df[["dest", "To Bank"]].rename(columns={"dest": "node", "To Bank": "bank"})
    nodes = (pd.concat([a.drop_duplicates("node"), b.drop_duplicates("node")])
               .drop_duplicates("node").reset_index(drop=True))
    node_id = pd.Series(np.arange(len(nodes), dtype=np.int64), index=nodes["node"])
    bank_code, banks = pd.factorize(nodes["bank"])
    x = torch.from_numpy(bank_code.astype(np.int64)).view(-1, 1)   # nn.Embedding trong model
    return node_id, x, len(banks)


def build_graph(name, node_id, x):
    ea = pd.read_parquet(EDGE_ATTR.format(name))
    si = node_id.reindex(ea["src"]).to_numpy()
    di = node_id.reindex(ea["dest"]).to_numpy()
    assert not (np.isnan(si).any() or np.isnan(di).any()), "cạnh có node ngoài vocab"

    feat = ea.drop(columns=["src", "dest"]).to_numpy(dtype=np.float32)
    fwd = torch.from_numpy(np.stack([si, di]).astype(np.int64))
    rev = fwd.flip(0)                                   # cạnh ngược, cùng edge_attr
    attr = torch.from_numpy(feat)
    flag = torch.zeros(attr.size(0), 1)

    data = Data(
        x=x,
        edge_index=torch.cat([fwd, rev], dim=1),
        edge_attr=torch.cat([torch.cat([attr, flag], 1),
                             torch.cat([attr, flag + 1], 1)], dim=0),   # +cờ is_reverse
        num_nodes=x.size(0),
    )
    deg = torch.bincount(data.edge_index[0], minlength=data.num_nodes)
    print(f"  {name:5s}: {data.num_nodes:,} node | {fwd.size(1):,} cạnh có hướng "
          f"-> {data.edge_index.size(1):,} sau cạnh ngược | "
          f"edge_attr {tuple(data.edge_attr.shape)} | node cô lập {int((deg == 0).sum()):,}")
    return data


def main():
    df = load_data()
    node_id, x, n_bank = build_vocab(df)
    print(f"vocab: {len(node_id):,} node | {n_bank:,} bank")

    graphs = {n: build_graph(n, node_id, x) for n in GRAPH_SPLITS}
    os.makedirs(os.path.dirname(GRAPHS_PT), exist_ok=True)
    torch.save({"graphs": graphs, "num_banks": n_bank}, GRAPHS_PT)

    # ánh xạ giao dịch -> (node gửi, node nhận), theo ĐÚNG thứ tự file gốc
    df = df.sort_values("ori_idx")
    txn = np.stack([node_id.reindex(df["src"]).to_numpy(),
                    node_id.reindex(df["dest"]).to_numpy()]).astype(np.int64).T
    np.save(TXN_NODES, txn)
    print(f"đã lưu {GRAPHS_PT} và {TXN_NODES} shape={txn.shape}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/metrics.py
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve, roc_curve,
)

from paths import RESULTS_CSV

RESULTS_PATH = RESULTS_CSV

COLUMNS = ["model", "split", "time", "f1_minority", "precision", "recall",
           "pr_auc", "recall@fpr1%", "precision@1000", "threshold",
           "n", "n_pos", "seed", "train_time_s", "stage", "params"]


def find_best_threshold(y_true, y_score):
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    best = np.argmax(f1[:-1])
    return float(thr[best])


def recall_at_fpr(y_true, y_score, max_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.interp(max_fpr, fpr, tpr))


def precision_at_k(y_true, y_score, k=1000):
    order = np.argsort(y_score)[::-1][:k]
    return float(np.asarray(y_true)[order].mean())


def evaluate(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)
    return {
        "f1_minority": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score),
        "recall@fpr1%": recall_at_fpr(y_true, y_score, 0.01),
        "precision@1000": precision_at_k(y_true, y_score, 1000),
        "threshold": threshold,
        "n": len(y_true),
        "n_pos": int(y_true.sum()),
    }


def evaluate_val_test(y_val, s_val, y_test, s_test):
    """Quy trinh chuan: tune threshold tren val -> danh gia ca val va test."""
    thr = find_best_threshold(y_val, s_val)
    return {
        "val": evaluate(y_val, s_val, thr),
        "test": evaluate(y_test, s_test, thr),
    }


def log_result(model, split, metrics, path=RESULTS_PATH, **extra):
    row = {"model": model, "split": split,
           "time": pd.Timestamp.now().isoformat(timespec="seconds"),
           **metrics, **extra}
    df = pd.DataFrame([row]).reindex(columns=COLUMNS)
    df.to_csv(path, mode="a", index=False, header=not os.path.exists(path))

In [ ]:
%%writefile /kaggle/working/train_gnn.py
import json, os, sys, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch_geometric.nn import MessagePassing

from metrics import find_best_threshold, evaluate, log_result
from paths import TX_FEAT, TXN_NODES, GRAPHS_PT, TXN_MATRIX, SCORES_DIR

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [0, 1, 2]
HID, NEG_RATIO, EPOCHS, BATCH, LR, PATIENCE = 32, 50, 200, 32768, 3e-3, 20


class EdgeSAGE(MessagePassing):
    """SAGE có edge_attr: message = MLP([h_j ‖ edge_attr]), aggr=mean, + nhánh root."""
    def __init__(self, in_dim, edge_dim, out_dim):
        super().__init__(aggr="mean")
        self.msg = nn.Linear(in_dim + edge_dim, out_dim)
        self.root = nn.Linear(in_dim, out_dim)

    def forward(self, h, edge_index, edge_attr):
        return self.root(h) + self.propagate(edge_index, x=h, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        return F.relu(self.msg(torch.cat([x_j, edge_attr], dim=-1)))


class Model(nn.Module):
    def __init__(self, num_banks, edge_dim, tx_dim, hid=HID):
        super().__init__()
        self.bank = nn.Embedding(num_banks, hid)
        self.c1 = EdgeSAGE(hid, edge_dim, hid)
        self.c2 = EdgeSAGE(hid, edge_dim, hid)
        self.head = nn.Sequential(
            nn.Linear(2 * hid + tx_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))

    def encode(self, g):
        h = self.bank(g.x.squeeze(-1))
        h = F.relu(self.c1(h, g.edge_index, g.edge_attr))
        return self.c2(h, g.edge_index, g.edge_attr)

    def forward(self, h, pairs, tx):
        return self.head(torch.cat([h[pairs[:, 0]], h[pairs[:, 1]], tx], -1)).squeeze(-1)


def load_all(no_edge_attr=False):
    split = pd.read_parquet(TX_FEAT, columns=["split"])["split"].to_numpy()
    nodes = np.load(TXN_NODES)
    blob = torch.load(GRAPHS_PT, weights_only=False)
    graphs, num_banks = blob["graphs"], blob["num_banks"]

    d = {}
    for s in ["train", "val", "test"]:
        m = pd.read_parquet(TXN_MATRIX.format(s))
        y = m.pop("Is Laundering").to_numpy().astype("float32")
        d[s] = [m.to_numpy("float32"), y, torch.from_numpy(nodes[split == s])]

    sc = StandardScaler().fit(d["train"][0])                 # fit CHỈ trên train
    for s in d:
        d[s][0] = sc.transform(d[s][0]).astype("float32")

    for s, g in graphs.items():
        if no_edge_attr:
            g.edge_attr = torch.zeros_like(g.edge_attr)      # đối chứng topology-only
        graphs[s] = g.to(DEV)
    return d, graphs, num_banks


def sample_epoch(y, rng):
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    neg = rng.choice(neg, size=min(len(neg), NEG_RATIO * len(pos)), replace=False)
    idx = np.concatenate([pos, neg]); rng.shuffle(idx)
    return idx


@torch.no_grad()
def score(model, g, X, pairs, chunk=200_000):
    model.eval()
    h = model.encode(g)
    out = [torch.sigmoid(model(h, pairs[i:i + chunk].to(DEV),
                               torch.from_numpy(X[i:i + chunk]).to(DEV))).float().cpu().numpy()
           for i in range(0, len(X), chunk)]
    return np.concatenate(out), h


def run(seed, d, graphs, num_banks, tag):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    Xtr, ytr, ptr = d["train"]
    model = Model(num_banks, graphs["train"].edge_attr.size(1), Xtr.shape[1]).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    amp = DEV.type == "cuda"
    gscaler = torch.amp.GradScaler("cuda", enabled=amp)

    best, best_state, bad, t0 = -1, None, 0, time.time()
    for ep in range(EPOCHS):
        idx = sample_epoch(ytr, rng)
        model.train()
        for i in range(0, len(idx), BATCH):
            b = idx[i:i + BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=amp):
                h = model.encode(graphs["train"])            # full-graph: 515K node, đủ nhỏ
                logit = model(h, ptr[b].to(DEV), torch.from_numpy(Xtr[b]).to(DEV))
                loss = F.binary_cross_entropy_with_logits(
                    logit, torch.from_numpy(ytr[b]).to(DEV))
            gscaler.scale(loss).backward(); gscaler.step(opt); gscaler.update()

        s_val, _ = score(model, graphs["val"], d["val"][0], d["val"][2])
        pr = evaluate(d["val"][1], s_val, 0.5)["pr_auc"]
        if pr > best:
            best, bad = pr, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
        if ep % 10 == 0:
            print(f"  ep{ep:3d} loss={loss.item():.4f} val_pr_auc={pr:.4f} (best {best:.4f})")

    model.load_state_dict(best_state)
    train_time_s = round(time.time() - t0, 1)

    s_val, h_val = score(model, graphs["val"], d["val"][0], d["val"][2])
    s_test, h_test = score(model, graphs["test"], d["test"][0], d["test"][2])
    thr = find_best_threshold(d["val"][1], s_val)
    for s, sc_ in [("val", s_val), ("test", s_test)]:
        log_result(tag, s, evaluate(d[s][1], sc_, thr), seed=seed, stage="final",
                   train_time_s=train_time_s,
                   params=json.dumps({"hid": HID, "neg_ratio": NEG_RATIO, "epochs_run": ep + 1}))
        np.save(os.path.join(SCORES_DIR, f"{tag}_seed{seed}_{s}.npy"), sc_)

    # embedding cho nhóm 3 (graph lũy tiến, không dùng nhãn test)
    np.save(os.path.join(SCORES_DIR, f"emb_{tag}_seed{seed}_val.npy"), h_val.float().cpu().numpy())
    np.save(os.path.join(SCORES_DIR, f"emb_{tag}_seed{seed}_test.npy"), h_test.float().cpu().numpy())
    print(f"[{tag} seed{seed}] {train_time_s}s | val_pr_auc={best:.4f}")


def main(mode="edge"):
    os.makedirs(SCORES_DIR, exist_ok=True)
    tag = "sage" if mode == "edge" else "sage_noedge"
    print(f"[mode={mode}] tag={tag} | device={DEV}")
    d, graphs, num_banks = load_all(no_edge_attr=(mode != "edge"))
    for seed in SEEDS:
        run(seed, d, graphs, num_banks, tag)


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "edge")

## 2. Kiểm tra đường dẫn

Phải in ra `TRANS_CSV tồn tại: True` trước khi chạy tiếp. Nếu False thì chưa Add Data đúng dataset.

In [ ]:
import os, subprocess, sys

W = "/kaggle/working"
py = sorted(f for f in os.listdir(W) if f.endswith(".py"))
print(f"{len(py)}/9 file .py:", py)
assert len(py) == 9, "Thiếu file -> chạy lại toàn bộ các cell %%writefile ở trên"

# vì sao có ModuleNotFoundError: xem Python có tự thêm thư mục script vào sys.path không
open("/tmp/_probe.py", "w").write("import sys; print('sys.path[0] =', repr(sys.path[0]))")
r = subprocess.run([sys.executable, "/tmp/_probe.py"], capture_output=True, text=True, cwd="/")
print(r.stdout.strip(), "| PYTHONSAFEPATH =", os.environ.get("PYTHONSAFEPATH"))

env = dict(os.environ, PYTHONPATH=W)
r = subprocess.run([sys.executable, "-c", "import paths; print('import paths OK')"],
                   capture_output=True, text=True, cwd="/", env=env)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/paths.py

## 3. Chia train/val/test theo thời gian (60/20/20)

`temporal_split_index.py` — ~2 phút

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/temporal_split_index.py

## 4. Feature mức giao dịch

`feature_transaction.py` — ~3 phút

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/feature_transaction.py

## 5. Feature as-of mức node và cặp (src, dest)

`feature_node.py` — ~15-25 phút

**Đây là đỉnh RAM (~12GB).** Kernel chết ở bước này nghĩa là accelerator đang là P100 (13GB RAM) chứ không phải T4 x2 (29GB) — đổi trong Settings rồi chạy lại.

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/feature_node.py

## 6. Feature cạnh gộp theo cửa sổ lũy tiến + kiểm tra leak

`feature_edge.py` — ~10-15 phút

Cell này in `PASS: edge_attr val bất biến khi xáo dữ liệu val` — đó là kiểm tra rò rỉ, không được bỏ qua.

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/feature_edge.py

## 7. Ghép thành ma trận 95 feature cho từng split

`assemble_txn.py` — ~3 phút

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/assemble_txn.py

## 8. Dựng 3 graph PyG lũy tiến

`build_graph.py` — ~5 phút

In [ ]:
!PYTHONPATH=/kaggle/working python /kaggle/working/build_graph.py

## 9. Train GNN

Hai lần chạy, mỗi lần 3 seed. Kiểm tra dòng đầu in đúng `tag=sage` và `tag=sage_noedge` — điều kiện trong code là `mode != "edge"` nên gõ sai chữ sẽ âm thầm chạy noedge chứ không báo lỗi.

In [ ]:
# CÓ edge_attr
!PYTHONPATH=/kaggle/working python /kaggle/working/train_gnn.py edge

In [ ]:
# Đối chứng topology-only: edge_attr bị zero hoá
!PYTHONPATH=/kaggle/working python /kaggle/working/train_gnn.py noedge

## 10. Kết quả

In [ ]:
import pandas as pd

r = pd.read_csv("/kaggle/working/results.csv")
print(r.groupby(["model", "split"])[["pr_auc", "f1_minority", "recall@fpr1%"]]
       .agg(["mean", "std"]).round(4))
r.tail(12)

## 11. Kích thước output

`/kaggle/working` giới hạn 20GB và Save Version sẽ rất lâu nếu giữ hết. Xoá thủ công file nào không cần tải về.

In [ ]:
import os

total = 0
for root, _, files in os.walk("/kaggle/working"):
    for f in files:
        p = os.path.join(root, f)
        sz = os.path.getsize(p)
        total += sz
        if sz > 50e6:
            print(f"{sz/1e6:8.0f} MB  {p}")
print(f"--------\n{total/1e9:8.2f} GB tổng cộng")